# Generalised code for finding NSC


In [3]:
import numpy as np
import pandas, os, astropy, scipy, math
import matplotlib.pyplot as plt
import sklearn
from photutils.aperture import CircularAperture, SkyCircularAperture, aperture_photometry, EllipticalAperture, SkyEllipticalAperture
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.wcs import WCS
from astropy.visualization import simple_norm
from astropy.wcs.utils import pixel_to_skycoord
import astropy.io.fits as fits
from reproject import reproject_interp
import img_scale
from mpl_toolkits.axes_grid1 import Divider, Size
from astropy.cosmology import FlatLambdaCDM 
from astropy.wcs.utils import proj_plane_pixel_scales
import glob

# Doing the non-plotting thingies

In [ ]:
#Loading the Data
galcenx = 
galceny =  
cut_cat = 
hdul = 
re_arcsec = 
distgalpc = 
newconcent = cut_cat['newconcent']

In [ ]:
#Applying cuts to the data
# Define cuts that we are using(comment out this cell to get the original data)
use_cat = cut_cat

max_mag = 35.  # Maxmum mag (Telescope cut)
snr_thresh = 3.  # For nearby, 5
# crowd_thresh = 0.5 #[mag], Normal cut  
# sharp_thresh = 0.25 #Normal cut  defraction spikes
# round_thresh = 3.  #Weak constraint (elongated)
flag_thresh = np.array([0, 2])
flag_label = np.array(['Good', 'Edge', 'Bad,Sat-pixel', 'Edge+badpixel'])

## Instrumental VEGAMAG magnitude cut
print(f"Total dolphot catalogue length: {len(use_cat)}")
print(f"Require: mag < {max_mag}")
condition = (use_cat.iloc[:,16] < max_mag) & (use_cat.iloc[:,29] < max_mag)
sub_dolcut_cat = use_cat[condition]
print(f"New catalogue length: {len(sub_dolcut_cat)}")

## SNR cut
print(f"Require: S/N > {snr_thresh}.")
condition = (sub_dolcut_cat.iloc[:,20] > snr_thresh) & (sub_dolcut_cat.iloc[:,33] > snr_thresh)
sub_dolcut_cat = sub_dolcut_cat[condition]
print(f"New catalogue length: {len(sub_dolcut_cat)}")

## Object Type cut
print("Remove sources that are not type point-like.")
condition = (sub_dolcut_cat.iloc[:,11] <= 2)
sub_dolcut_cat = sub_dolcut_cat[condition]
print(f"New catalogue length: {len(sub_dolcut_cat)}")

## Photometry quality flag cut
print("Remove sources that have flag= ", flag_thresh, "or", flag_label[flag_thresh])
condition = np.isin(sub_dolcut_cat.iloc[:,24], flag_thresh) & np.isin(sub_dolcut_cat.iloc[:,37], flag_thresh)
sub_dolcut_cat = sub_dolcut_cat[condition]
print(f"New catalogue length: {len(sub_dolcut_cat)}")

cut_cat = sub_dolcut_cat

In [ ]:
# Writing a function to calculate the magnitude difference bewteen the 2 largest objects
def max_mag_diff(magar):
    lstmag = magar.tolist()
    sortedlstmag = sorted(lstmag)
    magdif1 = sortedlstmag[0] - sortedlstmag[1]
    print(" The magnitude difference between the brightest and 2nd brightest object is {}".format(magdif1))

In [ ]:
#creating 2 numpy arrays that hold the xc and yc of each bright object
xcgal = cut_cat['xc'].to_numpy()
ycgal = cut_cat['yc'].to_numpy()
f475mag = cut_cat['F475W_mag_vega'].to_numpy()
f814mag = cut_cat['F814W_mag_vega'].to_numpy()

In [ ]:
#calculating the distance from the centre of the galaxy for each bright object
distgal = np.sqrt((xcgal - galcenx)**2 + (ycgal - galceny)**2)

In [ ]:
#Converting the re from arcsec to pixels
if len(hdul) > 1:
    hdr = hdul[1].header
else:
    hdr = hdul[0].header
wcs = WCS(hdr)
pixel_scale_degrees = proj_plane_pixel_scales(wcs)
scale_arcsec_per_pixel = pixel_scale_degrees[0] * 3600
scale_pixel_per_arcsec = 1 / scale_arcsec_per_pixel
re_pixels = re_arcsec * scale_pixel_per_arcsec
print(re_pixels)

In [ ]:
#refining the bright object list to only include objects within re and finding the colour of the objects
distgal_re = distgal[distgal <= re_pixels]
f475mag_re = f475mag[distgal <= re_pixels]
f814mag_re = f814mag[distgal <= re_pixels]
color = f475mag_re - f814mag_re
colorfull = f475mag - f814mag

In [ ]:
# normalising the distance in terms of re
distgal_re_n = distgal_re/re_pixels

In [ ]:
#Converting into absolute Mag

absmag_re = f814mag_re - 5*np.log10(distgalpc/10)

In [ ]:
#finding the mag of the object in 0.2 re
repixelsp2 = re_pixels * 0.2
print(repixelsp2)
f814mag_rep2 = f814mag[distgal <= repixelsp2]

In [ ]:
#Finding the mag difference of objects in 1 re then 0.2 re
max_mag_diff(f475mag_re)# 1 re
max_mag_diff(f814mag_rep2)# 0.2 re

In [ ]:
# accessing the fits file to get the header and the wcs information
F814W = hdul[1].data
F814Wheader = hdul[1].header

In [ ]:
# Writing code to find the index of the brightest object in 1 re 
minmag = 100
index = -1

for i in range(len(f814mag_re)):
    if f814mag_re[i] < minmag and lower < color[i] < higher and lower < newconcent[i] < higher and distgal_re_n[i] < 0.2:
        minmag = f814mag_re[i]
        index = i

if index == -1:
    print("No valid object found")
else:
    concentration = newconcent[index]
    colorval = color[index]
    f814magval = f814mag_re[index]
    objxc = xcgal[index]
    objyc = ycgal[index]
    objdist = distgal_re[index]
    print(index, minmag, concentration, colorval)

# Plotting all the plots

In [ ]:
#plotting distance vs magnitude for Bluer filter
plt.figure(figsize=(8,6))
plt.scatter(distgal, f475mag, color='blue', s=10)
plt.title('Distance from Galaxy Centre vs Bluer Filters Magnitude', fontsize=16)
plt.xlabel('Distance from Galaxy Centre (pixels)', fontsize=14)
plt.ylabel("Bluer Filters' Magnitude (Vega)", fontsize=14)
plt.grid()
plt.show()

In [ ]:
#plotting distance vs magnitude for Redder filter
plt.figure(figsize=(8,6))
plt.scatter(distgal, f814mag, color='red', s=10)
plt.title('Distance from Galaxy Centre vs Redder Filters Magnitude', fontsize=16)
plt.xlabel('Distance from Galaxy Centre (pixels)', fontsize=14)
plt.ylabel("Redder Filters' Magnitude (Vega)", fontsize=14)
plt.grid()
plt.show()

In [ ]:
#plotting distance vs magnitude for both filters together
plt.figure(figsize=(8,6))
plt.scatter(distgal, f475mag, color='blue', s=10, label='Bluer Filter')
plt.scatter(distgal, f814mag, color='red', s=10, label='Redder Filter')
plt.title('Distance from Galaxy Centre vs Magnitude', fontsize=16)
plt.xlabel('Distance from Galaxy Centre (pixels)', fontsize=14)
plt.ylabel('Magnitude (Vega)', fontsize=14)
plt.legend()
plt.grid()
plt.show()

In [ ]:
#plotting distance vs magnitude for Bluer Filter in re 
plt.figure(figsize=(8,6))
plt.scatter(distgal_re, f475mag_re, color='blue', s=10)
plt.title('Distance from Galaxy Centre vs Bluer Filters Magnitude (within re)', fontsize=16)
plt.xlabel('Distance from Galaxy Centre (pixels)', fontsize=14)
plt.ylabel("Bluer Filters' Magnitude (Vega)", fontsize=14)
plt.grid()
plt.show()  

In [ ]:
#plotting distance vs magnitude for Redder Filter in re
plt.figure(figsize=(8,6))
plt.scatter(distgal_re, f814mag_re, color='red', s=10)
plt.title('Distance from Galaxy Centre vs Redder Filters Magnitude (within re)', fontsize=16)
plt.xlabel('Distance from Galaxy Centre (pixels)', fontsize=14)
plt.ylabel("Redder Filters' Magnitude (Vega)", fontsize=14)
plt.grid()
plt.show()  

In [ ]:
#plotting distance vs magnitude for the objects within re for both filters together
plt.figure(figsize=(8,6))
plt.scatter(distgal_re, f475mag_re, color='blue', s=10, label='F475W')
plt.scatter(distgal_re, f814mag_re, color='red', s=10, label='F814W')
plt.title('Distance from Galaxy Centre vs Magnitude', fontsize=16)
plt.xlabel('Distance from Galaxy Centre (pixels)', fontsize=14)
plt.ylabel('Magnitude (Vega)', fontsize=14)
plt.legend()
plt.grid()
plt.show()

In [ ]:
#plotting a color vs magnitude for the objects within a re
plt.figure(figsize=(8,6))
plt.scatter(colorfull, f814mag, c = distgal,cmap = 'grey', s=10, alpha = 0.5)
plt.scatter(color, f814mag_re, c = distgal_re_n,cmap = 'jet', s=10)
plt.colorbar()
plt.scatter(colorval, f814magval, ec = 'red', fc = 'none', s=300, alpha = 1, label='object with lowest mag within re')
plt.vlines(x = 0.4, ymin = 19, ymax = 28, color = 'red', linestyle = '--', label = 'Color = 0.4')
plt.vlines(x = 1.5, ymin = 19, ymax = 28, color = 'red', linestyle = '--', label = 'Color = 1.5')
plt.title('Color of the objects vs F814W Magnitude (within re)', fontsize=16)
plt.xlabel('Color of the objects', fontsize=14) 
plt.ylabel('F814W Magnitude (Vega)', fontsize=14)
plt.grid()
plt.show()

In [ ]:
#Plotting the concentration vs Absolute magnitude graph withing 1 re
plt.figure(figsize=(8,6))
plt.scatter(newconcent, f814mag, c = distgal,cmap = 'grey', s=10, alpha = 0.5)
plt.scatter(newconcent[distgal <= re_pixels], f814mag_re, c = distgal_re_n,cmap = 'jet', s=10)
plt.colorbar()
plt.scatter(concentration, f814magval, ec = 'red', fc = 'none', s=300, alpha = 1, label='object with lowest mag within re')
plt.vlines(x = 0.3, ymin = 19, ymax = 28, color = 'red', linestyle = '--', label = 'Concentration = 0.3')
plt.vlines(x = 0.85, ymin = 19, ymax = 28, color = 'red', linestyle = '--', label = 'Concentration = 0.85')
plt.title('Concentration vs F814W Magnitude', fontsize=16)
plt.xlabel('Concentration', fontsize=14) 
plt.ylabel('Absolute Magnitude', fontsize=14)
plt.grid()
plt.show()

In [ ]:
#Plotting the F814W image for the galaxy and overplot all the bright objects within re

fig0 = plt.figure(figsize=(10,10))
plt.imshow(F814W, origin='lower', cmap='bone_r', vmin = -0.025, vmax = 0.025)
plt.colorbar()
plt.scatter(xcgal[distgal <= re_pixels], ycgal[distgal <= re_pixels], color='red', s=10, alpha = 0.5, label='Bright objects within re')
plt.scatter(objxc, objyc, ec = 'yellow', fc = 'none', s=300, alpha = 1, label='object with lowest mag within re')
plt.title('F814W Image of the Galaxy', fontsize=16)
plt.xlabel('Pixel X', fontsize=14)
plt.ylabel('Pixel Y', fontsize=14)
plt.grid()
plt.show()

In [ ]:
# zooming in for better visualization of the bright objects within re
fig0 = plt.figure(figsize=(10,10))
plt.imshow(F814W, origin='lower', cmap='bone_r', vmin = -0.025, vmax = 0.025)
plt.colorbar()
plt.scatter(xcgal[distgal <= re_pixels], ycgal[distgal <= re_pixels], color='red', s=10, label='Bright objects within re')
plt.scatter(objxc, objyc, ec = 'yellow', fc = 'none', s=300, alpha = 1, label='object with lowest mag within re')
plt.scatter(galcenx, galceny, color='green', s=50, alpha = 1, label='Galaxy centre')
plt.title('F814W Image of the Galaxy', fontsize=16)
plt.xlabel('X pixels', fontsize=14)
plt.ylabel('Y Pixels', fontsize=14)
plt.xlim(round(galcenx - 500), round(galcenx + 500))
plt.ylim(round(galceny - 250), round(galceny + 250))
plt.grid()
plt.show()

In [ ]:
# zooming in even more to see the brightest and closest object to the centre of the galaxy
fig0 = plt.figure(figsize=(10,10))
plt.imshow(F814W, origin='lower', cmap='bone_r', vmin = -0.025, vmax = 0.025)
plt.colorbar()
plt.scatter(xcgal[distgal <= re_pixels], ycgal[distgal <= re_pixels], color='red', s=30, alpha = 0.5, label='Bright objects within re')
plt.scatter(objxc, objyc, ec = 'yellow', fc = 'none', s=300, alpha = 1, label='object with lowest mag within re')
plt.scatter(galcenx, galceny, color='green', s=50, alpha = 1, label='Galaxy centre')
plt.title('F814W Image of the Galaxy', fontsize=16)
plt.xlabel('X pixels', fontsize=14)
plt.ylabel('Y pixels', fontsize=14)
plt.xlim(objxc-20, objxc+20)
plt.ylim(objyc-20, objyc+20)
plt.legend()
plt.grid()
plt.show()

In [ ]:
# trying to see how bright it is
fig0 = plt.figure(figsize=(10,10))
plt.imshow(F814W, origin='lower', cmap='bone_r', vmin = 0, vmax = 1)
plt.colorbar()
plt.scatter(xcgal[distgal <= re_pixels], ycgal[distgal <= re_pixels], color='red', s=30, alpha = 0.5, label='Bright objects within re')
plt.scatter(objxc, objyc, ec = 'yellow', fc = 'none', s=300, alpha = 1, label='object with lowest mag within re')
plt.scatter(galcenx, galceny, color='green', s=50, alpha = 1, label='Galaxy centre')
plt.title('F814W Image of the Galaxy', fontsize=16)
plt.xlabel('X pixel', fontsize=14)
plt.ylabel('Y pixel', fontsize=14)
plt.xlim(objxc-20, objxc+20)
plt.ylim(objyc-20, objyc+20)
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Zooming in even more
lim = round(objdist)
fig0 = plt.figure(figsize=(10,10))
plt.imshow(F814W, origin='lower', cmap='bone_r', vmin = -0.3, vmax = 2)
plt.colorbar()
plt.scatter(xcgal[distgal <= re_pixels], ycgal[distgal <= re_pixels], color='red', s=30, alpha = 0.5, label='Bright objects within re')
plt.scatter(objxc, objyc, ec = 'yellow', fc = 'none', s=300, alpha = 1, label='object with lowest mag within re')
plt.scatter(galcenx, galceny, color='green', s=50, alpha = 1, label='Galaxy centre')
plt.title('F814W Image of the Galaxy', fontsize=16)
plt.xlabel('X pixel', fontsize=14)
plt.ylabel('Y pixel', fontsize=14)
plt.xlim(objxc-lim, objxc+lim)
plt.ylim(objyc-lim, objyc+lim)
plt.legend()
plt.grid()
plt.show()